# Monte Carlo Analysis of CPPIThis notebook combines geometric Brownian motion with CPPI to study a distribution of possible wealth paths, terminal wealth and floor shortfalls. Simulations are model-dependent scenarios, not predictions.

In [ ]:
%matplotlib inlineimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport risk_kit as rkN_SCENARIOS = 500N_YEARS = 10MU = 0.07SIGMA = 0.15START = 100FLOOR = 0.80MULTIPLIER = 3RISK_FREE = 0.03SEED = 42

## 1. Simulate geometric Brownian motionThe shared simulator uses exact lognormal one-step returns. The first price row is the common starting value.

In [ ]:
prices = rk.gbm(N_YEARS, N_SCENARIOS, MU, SIGMA, s_0=START, seed=SEED)ax = prices.plot(legend=False, alpha=0.12, figsize=(11, 5))ax.set(title='Monte Carlo price paths', xlabel='Monthly time step', ylabel='Wealth index')plt.tight_layout()

## 2. Run CPPI across scenariosEach simulation is treated as a separate risky asset column. CPPI weights and account values are tracked independently across scenarios.

In [ ]:
sim_returns = rk.gbm(N_YEARS, N_SCENARIOS, MU, SIGMA, prices=False, seed=SEED)cppi = rk.run_cppi(sim_returns, riskfree_rate=RISK_FREE, m=MULTIPLIER,                   start=START, floor=FLOOR, steps_per_year=12)wealth = cppi['Wealth']terminal = wealth.iloc[-1]wealth.iloc[:, :50].plot(legend=False, alpha=0.25, figsize=(11, 5), title='First 50 CPPI wealth paths')plt.tight_layout()

## 3. Terminal wealth and floor riskThe terminal floor is the initial protected amount for the static-floor strategy. Expected shortfall below the floor is reported as a positive loss conditional on a breach.

In [ ]:
terminal_floor = START * FLOORbreaches = terminal < terminal_floorbreach_count = int(breaches.sum())breach_probability = breaches.mean()shortfalls = (terminal_floor - terminal[breaches])conditional_shortfall = shortfalls.mean() if breach_count else 0.0print(f'Mean terminal wealth:       {terminal.mean():.2f}')print(f'Median terminal wealth:     {terminal.median():.2f}')print(f'Floor breach probability:   {breach_probability:.2%}')print(f'Conditional shortfall:      {conditional_shortfall:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))wealth.iloc[:, :50].plot(ax=axes[0], legend=False, alpha=0.25)axes[0].axhline(terminal_floor, linestyle='--', color='black', label='Initial floor')axes[0].set(title='Sample wealth paths', xlabel='Monthly time step', ylabel='Wealth')axes[0].legend()terminal.plot.hist(ax=axes[1], bins=35, orientation='horizontal', edgecolor='white')axes[1].axhline(START, linestyle=':', color='black', label='Initial wealth')axes[1].axhline(terminal.mean(), linestyle='--', label='Mean')axes[1].axhline(terminal.median(), linestyle='-.', label='Median')axes[1].axhline(terminal_floor, linestyle='--', color='red', label='Floor')axes[1].set(title='Terminal wealth distribution', xlabel='Scenario count', ylabel='Terminal wealth')axes[1].legend()plt.tight_layout()

## 4. Parameter experimentRun the same experiment across alternative volatility assumptions. A fixed random seed improves comparability, although the simulated return distribution changes with volatility.

In [ ]:
rows = []for sigma in [0.10, 0.15, 0.20, 0.25]:    rets = rk.gbm(N_YEARS, N_SCENARIOS, MU, sigma, prices=False, seed=SEED)    result = rk.run_cppi(rets, riskfree_rate=RISK_FREE, m=MULTIPLIER,                         start=START, floor=FLOOR)    final = result['Wealth'].iloc[-1]    rows.append({'Annual volatility': sigma, 'Mean terminal wealth': final.mean(),                 'Median terminal wealth': final.median(),                 'Floor breach probability': (final < terminal_floor).mean()})pd.DataFrame(rows).set_index('Annual volatility')

## 5. Interactive exploration (optional)The following controls require ipywidgets. The core notebook above runs without interactive controls.

In [ ]:
try:    import ipywidgets as widgets    from IPython.display import displayexcept ImportError:    print('Install ipywidgets to enable interactive controls: pip install ipywidgets')else:    def explore_cppi(mu=0.07, sigma=0.15, multiplier=3.0, floor=0.8, scenarios=100):        rets = rk.gbm(10, scenarios, mu, sigma, prices=False, seed=SEED)        result = rk.run_cppi(rets, riskfree_rate=RISK_FREE, m=multiplier, start=START, floor=floor)        ax = result['Wealth'].plot(legend=False, alpha=0.18, figsize=(9, 4))        ax.axhline(START*floor, linestyle='--', color='black')        ax.set(title='Interactive CPPI scenarios', ylabel='Wealth')        plt.show()    display(widgets.interactive(explore_cppi, mu=(0.0, 0.2, 0.01), sigma=(0.05, 0.35, 0.025),                                multiplier=(1.0, 5.0, 0.5), floor=(0.5, 1.0, 0.05),                                scenarios=widgets.IntSlider(min=25, max=500, step=25, value=100)))

## Interpretation and limitationsThe breach probability is a Monte Carlo estimate conditional on the selected process and parameters. It is not a guarantee of real-world protection. GBM omits jumps, clustered volatility, fat tails, changing correlations, trading costs and execution delays. CPPI is subject to gap risk.